# XIAP HPO terms → UBERON / CL


Maps the <XIAP/CTLA>-deficiency HPO term list onto the HPO→UBERON / HPO→CL links from the ancestor-propagation run,
in two modes:

* **direct** – only links asserted directly on the HPO term ( from HPO term definitions)
* **propagation** – direct links plus propagated links (by going up to uncestor until the link is found) that passed review (`Review_Verdict == "ok"`)

Negated links are excluded in both modes.

In [1]:
from pathlib import Path
import pandas as pd
# mapping files can be downloaded from the following link: https://github.com/aybugealtay/OntoXRefExtractor/tree/main/data/outputs/ancestor_propagation/2026-10-05_180704/claude_reviews
MAPPING_FILES_DIR = Path("../../data/outputs/ancestor_propagation/2026-10-05_180704/claude_reviews")


FILES = {
    "uberon": MAPPING_FILES_DIR / "hpo_to_uberon_with_propagation_reviewed.csv",
    "cl":     MAPPING_FILES_DIR / "hpo_to_cl_with_propagation_reviewed.csv",
}
cols = ["ID", "HPO_Label", "Filler_IRI", "Filler_Label"]

def get_links(path):
    df = pd.read_csv(path)
    df["ID"] = df["HPO_IRI"].str.split("/").str[-1].str.replace("_", ":")
    not_negated = ~df["Negated"].fillna(False).astype(bool)
    is_direct = df["Link_Type"] == "direct"
    is_ok = df["Review_Verdict"] == "ok"
    return {
        "direct":      df[is_direct & not_negated][cols].drop_duplicates(["ID", "Filler_IRI"]),
        "propagation": df[(is_direct | is_ok) & not_negated][cols].drop_duplicates(["ID", "Filler_IRI"]),
    }

links = {onto: get_links(path) for onto, path in FILES.items()}

In [2]:
# XIAP_HPOs = pd.read_csv("XIAP_def_hpo_terms.txt", header=None, names=["X"], sep="\t")
# XIAP_HPOs[["Term", "ID"]] = XIAP_HPOs["X"].str.strip().str.extract(r"^(.*?)\s*\((HP:\d{7})\)$")
# XIAP_HPOs = XIAP_HPOs.drop(columns="X")
# assert XIAP_HPOs["ID"].notna().all(), "some lines didn't parse"
# XIAP_HPOs


TERM_FILES = {
    "XIAP": "XIAP_def_hpo_terms.txt",
    "CTLA": "CTLA_def_hpo_terms.txt",
    "FOXB3": "FOXB3_def_hpo_terms.txt"
}

def read_terms(path):
    t = pd.read_csv(path, header=None, names=["X"], sep="\t")
    t[["Term", "ID"]] = t["X"].str.strip().str.extract(r"^(.*?)\s*\((HP:\d{7})\)$")
    t = t.drop(columns="X")
    assert t["ID"].notna().all(), f"some lines in {path} didn't parse"
    return t

terms = {name: read_terms(path) for name, path in TERM_FILES.items()}

In [3]:
mapped = {
    (name, onto, mode): t.merge(l, on="ID", how="left")
    for name, t in terms.items()
    for onto, modes in links.items()
    for mode, l in modes.items()
}

summary = pd.DataFrame([
    {
        "gene": name,
        "ontology": onto,
        "mode": mode,
        "terms_mapped": m.dropna(subset=["Filler_IRI"])["ID"].nunique(),
        "terms_total": len(terms[name]),
        "links": m["Filler_IRI"].notna().sum(),
        "distinct_fillers": m["Filler_IRI"].nunique(),
    }
    for (name, onto, mode), m in mapped.items()
])
summary["coverage_%"] = (100 * summary.terms_mapped / summary.terms_total).round(1)
summary

,gene,ontology,mode,terms_mapped,terms_total,links,distinct_fillers,coverage_%
0,XIAP,uberon,direct,35,98,35,21,35.7
1,XIAP,uberon,propagation,42,98,42,23,42.9
2,XIAP,cl,direct,0,98,0,0,0.0
3,XIAP,cl,propagation,0,98,0,0,0.0
4,CTLA,uberon,direct,29,102,31,17,28.4
5,CTLA,uberon,propagation,39,102,41,19,38.2
6,CTLA,cl,direct,8,102,10,6,7.8
7,CTLA,cl,propagation,8,102,10,6,7.8
8,FOXB3,uberon,direct,24,104,26,21,23.1
9,FOXB3,uberon,propagation,26,104,28,23,25.0


In [4]:
for (name, onto, mode), m in mapped.items():
    print(f"\n=== {name} / {onto} / {mode} ===")
    display(m.groupby("Filler_Label")["ID"].nunique().sort_values(ascending=False))


=== XIAP / uberon / direct ===


Filler_Label
blood                          5
liver                          4
multicellular organism         4
intestine                      3
urine                          2
urinary bladder                2
pancreas                       1
superficial fascia             1
subcutaneous adipose tissue    1
stomach                        1
skin of body                   1
serous membrane                1
abdomen                        1
muscle organ                   1
lung                           1
left cardiac atrium            1
large intestine                1
kidney                         1
ileum                          1
blood serum                    1
lymph node                     1
Name: ID, dtype: int64


=== XIAP / uberon / propagation ===


Filler_Label
liver                          5
blood                          5
multicellular organism         4
intestine                      3
abdomen                        2
urine                          2
urinary bladder                2
ileum                          2
kidney                         2
lung                           2
serous membrane                1
superficial fascia             1
subcutaneous adipose tissue    1
stomach                        1
skin of body                   1
lymph node                     1
pancreas                       1
muscle organ                   1
left cardiac atrium            1
large intestine                1
feces                          1
blood serum                    1
vein                           1
Name: ID, dtype: int64


=== XIAP / cl / direct ===


...

Filler_Label
blood                                         4
muscle organ                                  2
multicellular organism                        2
abdomen                                       1
interatrial septum                            1
skin of body                                  1
posterior fontanelle                          1
pleural cavity                                1
lung                                          1
liver                                         1
intestine                                     1
interventricular septum                       1
heart                                         1
alimentary part of gastrointestinal system    1
fontanelle                                    1
feces                                         1
face                                          1
corpus callosum                               1
colon                                         1
cerebral hemisphere white matter              1
blood vessel               


=== FOXB3 / cl / direct ===


Filler_Label
erythrocyte    1
lymphocyte     1
Name: ID, dtype: int64


=== FOXB3 / cl / propagation ===


Filler_Label
erythrocyte    1
lymphocyte     1
Name: ID, dtype: int64

In [5]:
from pathlib import Path

OUT_DIR = Path("mappings")
OUT_DIR.mkdir(exist_ok=True)

for (name, onto, mode), m in mapped.items():
    prefix = onto.upper()
    out = m.dropna(subset=["Filler_IRI"]).copy()
    out[f"{prefix}_ID"] = out["Filler_IRI"].str.split("/").str[-1].str.replace("_", ":")
    out = out.drop(columns="HPO_Label").rename(columns={"Term": "HPO_Label", "ID": "HPO_ID", "Filler_Label": f"{prefix}_Label"})
    out = out[["HPO_Label", "HPO_ID", f"{prefix}_Label", f"{prefix}_ID"]]
    out.to_csv(OUT_DIR / f"{name}_hpo_to_{onto}_{mode}.csv", index=False)
    print(f"{name}/{onto}/{mode}: {len(out)} rows saved")

XIAP/uberon/direct: 35 rows saved
XIAP/uberon/propagation: 42 rows saved
XIAP/cl/direct: 0 rows saved
XIAP/cl/propagation: 0 rows saved
CTLA/uberon/direct: 31 rows saved
CTLA/uberon/propagation: 41 rows saved
CTLA/cl/direct: 10 rows saved
CTLA/cl/propagation: 10 rows saved
FOXB3/uberon/direct: 26 rows saved
FOXB3/uberon/propagation: 28 rows saved
FOXB3/cl/direct: 2 rows saved
FOXB3/cl/propagation: 2 rows saved
